In [1]:
import pandas as pd
from omicsqc.ingestion import ingest_data
from omicsqc.qc_layer.batch_metrics import batch_variation_test
from omicsqc.qc_layer.missingness_analysis import compute_missingness
from omicsqc.qc_layer.variation_test import compare_explained_variances
from omicsqc.qc_layer.qc_metrics import compute_qc_individual, compute_qc_pipeline

In [2]:
data_df = pd.read_csv("data/ProteinAbundances.csv")
meta_df = pd.read_csv("data/meta_data.csv")

data_df.head()

,Unnamed: 0,ba36_b01.126,ba36_b01.127N,ba36_b01.127C,ba36_b01.128N,ba36_b01.128C,ba36_b01.129N,ba36_b01.129C,ba36_b01.130N,ba36_b01.130C,...,rmap_b50.127N,rmap_b50.127C,rmap_b50.128N,rmap_b50.128C,rmap_b50.129N,rmap_b50.129C,rmap_b50.130N,rmap_b50.130C,rmap_b50.131N,rmap_b50.131C
0,|A6NIZ1,2.074253e+07,1.931603e+07,1.815059e+07,1.721684e+07,1.841878e+07,1.835115e+07,1.679205e+07,2.685073e+07,2.171703e+07,...,3.749850e+06,4.107925e+06,4.336841e+06,3.602929e+06,4.117088e+06,4.507979e+06,4.466593e+06,3.811035e+06,6.317644e+06,2.733053e+05
1,|A6NNZ2,1.045588e+09,1.053124e+09,1.197566e+09,1.028878e+09,9.813356e+08,8.888861e+08,1.249061e+09,1.078468e+09,1.113727e+09,...,2.475321e+08,3.096287e+08,3.000922e+08,2.666383e+08,3.296922e+08,2.951792e+08,3.098851e+08,2.724061e+08,4.118054e+08,1.234077e+07
2,|B2RBV5,1.905207e+04,1.036274e+04,2.408838e+04,1.770227e+04,3.011476e+04,1.830609e+04,1.698435e+04,2.884526e+04,2.811469e+04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,|P00761,1.923250e+08,1.507081e+08,1.790986e+08,2.183667e+08,2.198959e+08,2.655633e+08,1.259623e+08,2.029364e+08,2.246084e+08,...,2.309994e+07,3.011302e+07,2.623879e+07,2.726799e+07,2.636015e+07,2.735873e+07,2.849488e+07,2.564794e+07,9.068378e+07,2.202812e+06
4,|P02441,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Data Ingestion Test (checking data type)

In [3]:
protdata = ingest_data(data_df, meta_df,)

[Success] Found 558 overlapping technical samples.


In [4]:
print(type(protdata))
print(type(protdata.data))

<class 'omicsqc.models.ProteinData'>
<class 'pandas.DataFrame'>


### Missingness Analysis layer of QC

In [5]:
data_df, meta_df = protdata.data, protdata.metadata
missingness_report, missingness_logs = compute_missingness(data_df, meta_df)

In [6]:
missingness_report.overall_missingness

np.float64(22.33700568477125)

In [7]:
missingness_logs

'[Missingness Analysis]\n        Per-protein missingness (mean): 22.34%\n        Per-sample missingness (mean):  22.34%\n        Overall missingness:            22.34%\n        Spearman coefficient:           0.8788\n        Spearman p-value:               0.0000e+00\n        Inferred missingness nature: Evidence of intensity-dependent missingness, consistent with left-censoring'

### Batch variation metrics

In [8]:
data_df, meta_df = protdata.data, protdata.metadata
batch_metrics, batch_logs = batch_variation_test(data_df, meta_df, protein_threshold_pct= 40, )

In [9]:
batch_metrics.n_significant_pct

np.float64(88.52650944974076)

In [10]:
batch_logs

'[Batch Variation Test]\n    Statistical method:             anova \n\n    Significance level (alpha):     0.05 \n\n    Protein threshold:              40.00% \n\n    Significant proteins:           10586 \n\n    Significant proteins (%):       88.53% \n\n    Batch effect detected:          True \n\n    silhoutte score:                0.0101'

### Variance report

In [11]:
data_df, meta_df = protdata.data, protdata.metadata
variance_report, variance_logs = compare_explained_variances(data_df, meta_df, factors = ["Batch_overall", "JohnsonDiagnosis", "cogdx", "braaksc", "niareagansc", "msex"])

In [12]:
variance_report.comparison_df

,Batch_overall,JohnsonDiagnosis,cogdx,braaksc,niareagansc,msex
uniprot_id,,,,,,
|A6NIZ1,0.983358,0.038387,0.979518,0.083691,0.979549,0.075610
|A6NNZ2,0.982788,0.030591,0.940555,0.063946,0.942056,0.070171
|B2RBV5,0.921702,0.064289,0.140282,0.019911,0.183040,0.120300
|P00761,0.975779,0.042780,0.950320,0.085752,0.951167,0.067957
|P02441,0.162857,0.076206,0.189650,0.110207,0.080892,0.929225
...,...,...,...,...,...,...
ZXDC|Q2QGD7,0.919536,0.282548,0.904271,0.273552,0.895920,0.077667
ZYG11B|Q9C0D3,0.981044,0.019973,0.591913,0.042695,0.595232,0.029957
ZYX|Q15942,0.975822,0.015485,0.618144,0.036082,0.622609,0.032580


In [13]:
variance_logs

'[Explained Variance Comparison]\n    Factors analyzed:               Batch_overall, JohnsonDiagnosis, cogdx, braaksc, niareagansc, msex\n    Per-protein comparison shape:   (11958, 6)\n    Summary dataframe shape:        (6, 7)'

### Testing qc_layer pipeline

In [14]:
from time import perf_counter
start = perf_counter()
protdata = compute_qc_pipeline(protdata= protdata, data_df= protdata.data, meta_df= protdata.metadata, protein_threshold_pct= 60, 
                               variation_factors= ["Batch_overall", "JohnsonDiagnosis", "cogdx", "braaksc", "niareagansc", "msex"])
end = perf_counter()
print(end - start)

3.1032963999314234


In [15]:
protdata.logs

['[Missingness Analysis]\n        Per-protein missingness (mean): 22.34%\n        Per-sample missingness (mean):  22.34%\n        Overall missingness:            22.34%\n        Spearman coefficient:           0.8788\n        Spearman p-value:               0.0000e+00\n        Inferred missingness nature: Evidence of intensity-dependent missingness, consistent with left-censoring',
 '[Batch Variation Test]\n    Statistical method:             anova \n\n    Significance level (alpha):     0.05 \n\n    Protein threshold:              60.00% \n\n    Significant proteins:           10586 \n\n    Significant proteins (%):       88.53% \n\n    Batch effect detected:          True \n\n    silhoutte score:                0.0101',
 '[Explained Variance Comparison]\n    Factors analyzed:               Batch_overall, JohnsonDiagnosis, cogdx, braaksc, niareagansc, msex\n    Per-protein comparison shape:   (11958, 6)\n    Summary dataframe shape:        (6, 7)']

In [16]:
protdata.data.shape

(11958, 558)